In [1]:
import pandas as pd
import numpy as np
from gnomad_db.database import gnomAD_DB

In [2]:
# load data into a df
df = pd.read_csv('~/pcloud_sync/project/rescreen_gnomad_comparison/rescreen_raw_export.tsv', sep='\t', header=0, low_memory=False )


In [3]:
df['gene'] = [a.split('_')[0] for a in df['mut'].tolist()]
df['protein_consequence'] = [a.split('_')[1] for a in df['mut'].tolist()]
df['mutation_type'] = ['_'.join(a.split('_')[2:]) for a in df['mut'].tolist()]
df_key2=  df['chr'].map(str) + '_' + df['pos'].map(str) + '_' + df['ref'].map(str) + '_' + df['alt'].map(str)
df['key'] = df_key2
gene_dict = dict(zip(df.key, df.gene))
mutation_type_dict = dict(zip(df.key, df.mutation_type))
df_variants = df[['key','chr', 'pos', 'ref', 'alt']].drop_duplicates()

In [4]:
df_variants['gene'] = df_variants.key.map(gene_dict)
df_variants['mutation_type'] = df_variants.key.map(mutation_type_dict)

In [5]:
df_variants

,key,chr,pos,ref,alt,gene,mutation_type
0,10_115922774_G_A,10,115922774,G,A,C10orf118,missense
119,10_73051440_G_A,10,73051440,G,A,UNC5B,missense
131,11_1262312_G_A,11,1262312,G,A,MUC5B,missense
163,11_1264691_T_C,11,1264691,T,C,MUC5B,missense
352,11_1265858_C_T,11,1265858,C,T,MUC5B,missense
...,...,...,...,...,...,...,...
613891,9_100139669_G_GA,9,100139669,G,GA,NULL,NULL
613958,9_100139675_A_C,9,100139675,A,C,NULL,NULL
613988,9_123836943_G_A,9,123836943,G,A,NULL,NULL
613989,9_123836964_G_A,9,123836964,G,A,NULL,NULL


In [6]:
df_variants['mutation_type'].value_counts()

intron                             9522
missense                           4069
synonymous                         3175
NULL                                821
splice,intron                       471
3_prime_UTR                         263
5_prime_UTR                         180
non_coding_exon                     151
downstream                          102
splice,missense                      82
splice,synonymous                    77
upstream                             67
stop_gained                          41
frameshift                           24
inframe_deletion                     21
inframe_insertion                     9
splice_acceptor                       8
splice_donor                          7
stop_lost                             5
start_lost                            4
synonymous,NMD                        4
splice,frameshift                     2
NMD,missense                          2
splice,frameshift,intron              1
intron,splice_donor                   1


In [7]:
# download_link = "https://zenodo.org/record/5770384/files/gnomad_db_v2.1.1.sqlite3.gz?download=1"
# output_dir = "/path/to/local_data/gnomad_db"
# gnomAD_DB.download_and_unzip(download_link, output_dir)

In [8]:
# set up the gnomad database
database_location = "/path/to/local_data/gnomad_db/"
db = gnomAD_DB(database_location, genome="Grch38")

In [9]:
# check db columns, which we can query
# a few are not found in the database, so I remove them here. 
bad_columns = ["InbreedingCoeff", "VarDP", "AS_VQSLOD"]
db.columns = [a for a in db.columns if not (a in bad_columns)]

In [10]:
db.get_info_from_str("17:11784688:C>T", "AF")

0.00232514

In [11]:
# test queries
dummy_var_df = pd.DataFrame({
    "chrom": ["10", "9"], 
    "pos": [115922774, 100139675], 
    "ref": ["G", "A"], 
    "alt": ["A", "C"]})

# query from dataframe AF column
print(db.get_info_from_df(dummy_var_df, "AF"))
print('#######')
# query from dataframe AF and AF_popmax columns
print(db.get_info_from_df(dummy_var_df, "AF, AF_popmax"))
print('#######')
# query from dataframe all columns
print(db.get_info_from_df(dummy_var_df, "*"))
print('#######')
# query from string
print(db.get_info_from_str("21:9825790:C>T", "AF"))

# You can query also intervals of minor allele frequencies
db.get_info_for_interval(chrom=21, interval_start=9825780, interval_end=9825799, query="AF")

# You can pass a single string as a variant
db.get_info_from_str("21:9825790:C>T", "AF")
db.get_info_from_str("21:9825790:C>T", "*")

# You can look for the MAF scores in an interval
db.get_info_for_interval(chrom=21, interval_start=9025780, interval_end=9825799, query="*")

         AF
0  0.239124
1  0.045341
#######
         AF  AF_popmax
0  0.239124   0.328693
1  0.045341   0.064793
#######
  chrom        pos ref alt filter      AC       AN        AF    MQ     QD  \
0    10  115922774   G   A   PASS  7497.0  31352.0  0.239124  60.0  15.49   
1     9  100139675   A   C   PASS  1411.0  31120.0  0.045341  60.0  14.00   

   ReadPosRankSum  AC_popmax  AN_popmax  AF_popmax    AF_eas    AF_nfe  \
0           0.319     2857.0     8692.0   0.328693  0.065468  0.218941   
1           0.577      989.0    15264.0   0.064793  0.000000  0.064793   

     AF_fin    AF_afr    AF_asj  
0  0.204205  0.328693  0.341379  
1  0.043098  0.014296  0.131034  
#######
0.000353008


,chrom,pos,ref,alt,filter,AC,AN,AF,MQ,QD,ReadPosRankSum,AC_popmax,AN_popmax,AF_popmax,AF_eas,AF_nfe,AF_fin,AF_afr,AF_asj
0,21,9411211,A,G,RF,1.0,21060.0,0.000047,50.40,2.57,0.791,1.0,9948.0,0.000101,0.000000,0.000101,0.000000,0.000000,0.0
1,21,9411219,T,C,PASS,1.0,23588.0,0.000042,52.86,5.45,-0.634,1.0,11240.0,0.000089,0.000000,0.000089,0.000000,0.000000,0.0
2,21,9411223,T,A,RF,4.0,24938.0,0.000160,50.72,2.33,-0.035,4.0,7202.0,0.000555,0.000000,0.000000,0.000000,0.000555,0.0
3,21,9411239,G,A,RF,1.0,27786.0,0.000036,51.04,2.58,0.159,1.0,1402.0,0.000713,0.000713,0.000000,0.000000,0.000000,0.0
4,21,9411245,C,A,RF,20.0,28262.0,0.000708,53.96,3.94,-0.049,1.0,770.0,0.001299,0.000000,0.001024,0.000626,0.000376,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60053,21,9825796,C,CGCGT,RF,189.0,28418.0,0.006651,33.58,2.79,-0.297,17.0,684.0,0.024854,0.017129,0.004405,0.026466,0.000712,0.0
60054,21,9825796,C,CGT,AC0;RF,0.0,28418.0,0.000000,33.58,2.79,-0.297,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.0
60055,21,9825796,C,CGTGT,AC0;RF,0.0,28418.0,0.000000,33.58,2.79,-0.297,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.0
60056,21,9825796,C,T,RF,1.0,28418.0,0.000035,33.58,2.79,-0.297,1.0,8424.0,0.000119,0.000000,0.000000,0.000000,0.000119,0.0


In [20]:
# build a way to query all the variants
df_variants_query = df_variants[['chr', 'pos', 'ref', 'alt']]
df_variants_query.columns = ['chrom', 'pos', 'ref', 'alt']
# query the database and write back the result
df_AF = db.get_info_from_df(df_variants_query, "AF")
print(sum(np.isnan(df_AF.AF)))
df_variants['gnomad_AF'] = df_AF.values

2557


In [29]:
sum(np.isnan(df_variants.gnomad_AF))
sum(df_variants.gnomad_AF==0)

v1= "2:233676124:G>C"
v1= "10:95892036:G>A"
print(db.get_info_from_str("2:198266834:T>C", "*"))
print(db.get_info_from_str(v1, "*"))

db.get_info_for_interval(chrom=10, interval_start=95892035, interval_end=95892037, query="*")


chrom                     2
pos               198266834
ref                       T
alt                       C
filter                   RF
AC                      1.0
AN                  31396.0
AF                 0.000032
MQ                     60.0
QD                     2.98
ReadPosRankSum        0.829
AC_popmax               1.0
AN_popmax           15420.0
AF_popmax          0.000065
AF_eas                  0.0
AF_nfe             0.000065
AF_fin                  0.0
AF_afr                  0.0
AF_asj                  0.0
Name: 0, dtype: object
Empty DataFrame
Columns: [chrom, pos, ref, alt, filter, AC, AN, AF, MQ, QD, ReadPosRankSum, AC_popmax, AN_popmax, AF_popmax, AF_eas, AF_nfe, AF_fin, AF_afr, AF_asj]
Index: []


,chrom,pos,ref,alt,filter,AC,AN,AF,MQ,QD,ReadPosRankSum,AC_popmax,AN_popmax,AF_popmax,AF_eas,AF_nfe,AF_fin,AF_afr,AF_asj
0,10,95892035,C,T,PASS,2.0,31376.0,0.000064,60.0,15.23,0.634,1.0,848.0,0.001179,0.000000,0.0,0.0,0.000115,0.0
1,10,95892037,T,C,PASS,7.0,31350.0,0.000223,60.0,11.81,0.157,7.0,1560.0,0.004487,0.004487,0.0,0.0,0.000000,0.0


In [323]:
# get allele count
allele_map = {'0/1': 1, '1/1': 2}
df['allele_count'] = [allele_map[a[0:3]] for a in df.info3.tolist()]
df

,key,chr,pos,ref,alt,info1,info2,info3,file,trial,...,phred,patient,disease,AF_1000Gp_%,AF_cat,Phred20,mut,protein_consequence,mutation_type,allele_count
0,10_115922774_G_A,10,115922774,G,A,ADP=1011;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:1012:1011:538:472:46.69%:5.3487E-172:3...,DS-262253_006-02_Dx_SA_R1R2.bwa.mem.st.rmdup.var,CTP_001,...,0.001,006-02,AML,20.726800,AF_>1,<20,C10orf118_T85I_missense,T85I,missense,1
1,10_115922774_G_A,10,115922774,G,A,ADP=1037;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:1066:1037:564:473:45.61%:2.366E-171:47...,DS-381492_057-201_Dx_BS_R1R2.bwa.mem.st.rmdup.var,CTP_201,...,0.001,057-201,ET,20.726800,AF_>1,<20,C10orf118_T85I_missense,T85I,missense,1
2,10_115922774_G_A,10,115922774,G,A,ADP=1045;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:1053:1045:609:436:41.72%:1.0108E-154:3...,DS-262257_006-05_Dx_SA_R1R2.bwa.mem.st.rmdup.var,CTP_001,...,0.001,006-05,AML,20.726800,AF_>1,<20,C10orf118_T85I_missense,T85I,missense,1
3,10_115922774_G_A,10,115922774,G,A,ADP=104;WT=0;HET=0;HOM=1;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,1/1:255:104:104:2:102:98.08%:2.4482E-58:40:38:...,DS-228801_003-02_Dx_Pr_SA_R1R2.bwa.mem.var0.03...,CTP_001,...,0.001,003-02,AML,20.726800,AF_>1,<20,C10orf118_T85I_missense,T85I,missense,2
4,10_115922774_G_A,10,115922774,G,A,ADP=1072;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:1073:1072:552:518:48.32%:1.3357E-190:3...,DS-285197_004-007_Dx_BU_R1R2.bwa.mem.st.rmdup.var,CTP_101,...,0.001,004-007,AML,20.726800,AF_>1,<20,C10orf118_T85I_missense,T85I,missense,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613986,9_100139675_A_C,9,100139675,A,C,ADP=66;WT=0;HET=0;HOM=1;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,1/1:255:66:66:9:57:86.36%:3.328E-28:33:37:0:9:...,DS-259729_005-05_Dx_Pr_SA_R1R2.bwa.mem.st.rmdu...,CTP_001,...,NaN,005-05,AML,2.775560,AF_>1,NaN,NULL_NULLNULLNULL_NULL,NULLNULLNULL,NULL,2
613987,9_100139675_A_C,9,100139675,A,C,ADP=89;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:149:90:89:48:41:46.07%:1.0691E-15:42:41:14...,DS-381141_023-101_Dx_BS_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,NaN,023-101,MF,2.775560,AF_>1,NaN,NULL_NULLNULLNULL_NULL,NULLNULLNULL,NULL,1
613988,9_123836943_G_A,9,123836943,G,A,ADP=318;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:319:318:155:163:51.26%:3.7381E-62:38:3...,DS-288933_001-101_Dx_SA_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,NaN,001-101,MF,NaN,NaN,NaN,NULL_NULLNULLNULL_NULL,NULLNULLNULL,NULL,1
613989,9_123836964_G_A,9,123836964,G,A,ADP=63;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:110:68:63:33:30:47.62%:9.545E-12:52:46:23:...,DS-383737_034-102_Dx_BS_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,NaN,034-102,MF,1.617410,AF_>1,NaN,NULL_NULLNULLNULL_NULL,NULLNULLNULL,NULL,1


In [350]:
allele_count_agg = df.groupby('key').agg('sum')['allele_count']
pt_count_agg = df.groupby('key').agg('count')['allele_count']
df_variants['allele_count_case'] = allele_count_agg.loc[df_variants.key].values
df_variants['pt_count_case'] = pt_count_agg.loc[df_variants.key].values
df_variants['case_AF'] = df_variants['allele_count_case']/n_case_alleles
df_variants['case_fold_enrichment'] = df_variants['case_AF'] / df_variants['gnomad_AF']
df_variants_sorted = df_variants.sort_values('case_fold_enrichment',ascending=False, inplace=False)

/tmp/ipykernel_270220/847458501.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  allele_count_agg = df.groupby('key').agg('sum')['allele_count']


In [352]:
allele_count_agg = df.groupby('key').agg('sum')['allele_count']
allele_count_agg.loc[allele_count_agg.index=='2_32449878_T_A']

/tmp/ipykernel_270220/2141795657.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  allele_count_agg = df.groupby('key').agg('sum')['allele_count']


key
2_32449878_T_A    108
Name: allele_count, dtype: int64

In [353]:
pt_count_agg.loc[pt_count_agg.index=='2_32449878_T_A']

key
2_32449878_T_A    54
Name: allele_count, dtype: int64

In [354]:
df_variants_sorted.to_csv('/path/to/data/pcloud_sync/project/rescreen_gnomad_comparison/df_variants_with_gnomad_AF.tsv', sep='\t',index=False)

In [327]:
sum(np.isnan(df_variants.gnomad_AF))

2557

In [357]:
df.loc[df.key=='9_5073770_G_T']

,key,chr,pos,ref,alt,info1,info2,info3,file,trial,...,phred,patient,disease,AF_1000Gp_%,AF_cat,Phred20,mut,protein_consequence,mutation_type,allele_count
581427,9_5073770_G_T,9,5073770,G,T,ADP=136;WT=0;HET=0;HOM=1;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,1/1:255:136:136:24:112:82.35%:5.662E-53:48:52:...,DS-363471_060-111_Dx_PG_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,29.2,060-111,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,2
581428,9_5073770_G_T,9,5073770,G,T,ADP=1491;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:1574:1491:636:854:57.28%:0E0:34:35:311...,DS-362356_022-101_Dx_PG_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,29.2,022-101,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,1
581429,9_5073770_G_T,9,5073770,G,T,ADP=155;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:228:155:155:91:64:41.29%:1.3863E-23:54:54:...,DS-377789_060-107_Dx_BS_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,29.2,060-107,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,1
581430,9_5073770_G_T,9,5073770,G,T,ADP=174;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:177:174:98:76:43.68%:3.3083E-28:46:50:...,DS-381035_009-106_Dx_PG_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,29.2,009-106,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,1
581431,9_5073770_G_T,9,5073770,G,T,ADP=2401;WT=0;HET=0;HOM=1;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,1/1:255:2508:2401:234:2167:90.25%:0E0:34:35:11...,DS-361518_012-101_Dx_PG_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,29.2,012-101,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,2
581432,9_5073770_G_T,9,5073770,G,T,ADP=284;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:284:284:136:148:52.11%:9.3366E-57:54:5...,DS-363839_060-108_ITPD84_PG_R1R2.bwa.mem.st.rm...,CTP_102,...,29.2,060-108,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,1
581433,9_5073770_G_T,9,5073770,G,T,ADP=409;WT=0;HET=0;HOM=1;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,1/1:255:409:409:25:384:93.89%:5.6573E-205:49:5...,DS-363830_060-106_ITPD84_PG_R1R2.bwa.mem.st.rm...,CTP_102,...,29.2,060-106,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,2
581434,9_5073770_G_T,9,5073770,G,T,ADP=436;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:441:436:244:191:43.81%:5.6132E-70:54:5...,DS-389535_060-202_Dx_BS_R1R2.bwa.mem.st.rmdup.var,CTP_201,...,29.2,060-202,ET,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,1
581435,9_5073770_G_T,9,5073770,G,T,ADP=524;WT=0;HET=1;HOM=0;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,0/1:255:528:524:290:234:44.66%:6.567E-86:54:54...,DS-389540_060-204_Dx_BS_R1R2.bwa.mem.st.rmdup.var,CTP_201,...,29.2,060-204,ET,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,1
581436,9_5073770_G_T,9,5073770,G,T,ADP=574;WT=0;HET=0;HOM=1;NC=0,GT:GQ:SDP:DP:RD:AD:FREQ:PVAL:RBQ:ABQ:RDF:RDR:A...,1/1:255:574:574:94:480:83.62%:3.5794E-228:36:3...,DS-338795_007-105_Dx_PB_R1R2.bwa.mem.st.rmdup.var,CTP_102,...,29.2,007-105,MF,NaN,NaN,>20,JAK2_V617F_missense,V617F,missense,2


In [31]:
# Clinvar data: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
df_clinvar = pd.read_csv('/path/to/local_data/gnomad_db/variant_summary.txt', sep='\t', header=0, low_memory=False)

In [43]:
df_clinvar

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,key
7_4820844_GGAT_TGCTGTAAACTGTAACTGTAAA,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic,1,-,397704705,...,2,-,N,"ClinGen:CA215070,OMIM:613653.0001",3,2,4820844,GGAT,TGCTGTAAACTGTAACTGTAAA,7_4820844_GGAT_TGCTGTAAACTGTAACTGTAAA
7_4781213_GGAT_TGCTGTAAACTGTAACTGTAAA,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic,1,-,397704705,...,2,-,N,"ClinGen:CA215070,OMIM:613653.0001",3,2,4781213,GGAT,TGCTGTAAACTGTAACTGTAAA,7_4781213_GGAT_TGCTGTAAACTGTAACTGTAAA
7_4827360_GCTGCTGGACCTGCC_G,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,1,-,N,"ClinGen:CA215072,OMIM:613653.0002",1,3,4827360,GCTGCTGGACCTGCC,G,7_4827360_GCTGCTGGACCTGCC_G
7_4787729_GCTGCTGGACCTGCC_G,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,1,-,N,"ClinGen:CA215072,OMIM:613653.0002",1,3,4787729,GCTGCTGGACCTGCC,G,7_4787729_GCTGCTGGACCTGCC_G
15_85342440_G_A,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,1,-,N,"OMIM:613624.0001,ClinGen:CA210674,UniProtKB:Q9...",1,4,85342440,G,A,15_85342440_G_A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12_47986361_C_T,1952238,single nucleotide variant,NM_001844.5(COL2A1):c.1502G>A (p.Gly501Glu),1280,COL2A1,HGNC:2200,Likely pathogenic,1,"Jan 18, 2023",-1,...,1,-,N,-,2,1895438,47986361,C,T,12_47986361_C_T
17_40696002_A_G,1952239,single nucleotide variant,NM_000263.4(NAGLU):c.1978A>G (p.Asn660Asp),4669,NAGLU,HGNC:7632,Likely pathogenic,1,"Jan 16, 2023",-1,...,1,-,N,-,2,1895439,40696002,A,G,17_40696002_A_G
17_42543984_A_G,1952239,single nucleotide variant,NM_000263.4(NAGLU):c.1978A>G (p.Asn660Asp),4669,NAGLU,HGNC:7632,Likely pathogenic,1,"Jan 16, 2023",-1,...,1,-,N,-,2,1895439,42543984,A,G,17_42543984_A_G
19_15291863_G_T,1952240,single nucleotide variant,NM_000435.3(NOTCH3):c.2903C>A (p.Ser968Ter),4854,NOTCH3,HGNC:7883,Likely pathogenic,1,"Jan 19, 2023",-1,...,1,-,N,-,2,1895440,15291863,G,T,19_15291863_G_T


In [42]:
# make key column  10_115922774_G_A
kk = df_clinvar.Chromosome + "_" + df_clinvar.PositionVCF.astype('string') + "_" + df_clinvar.ReferenceAlleleVCF + "_" + df_clinvar.AlternateAlleleVCF
df_clinvar['key'] = kk
df_clinvar.index = kk 

In [68]:
# 5610 variants contained in clinVar
len(set(df_clinvar.key.values).intersection(df_variants.key.values))
df_clinvar_sub = df_clinvar.loc[set(df_clinvar.key.values).intersection(df_variants.key.values),]

/tmp/ipykernel_901866/2600330565.py:3: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  df_clinvar_sub = df_clinvar.loc[set(df_clinvar.key.values).intersection(df_variants.key.values),]


In [108]:
b = [df_clinvar_sub.ClinicalSignificance.get(a, 'NA') for a in df_variants.key.values]
df_variants.ClinicalSignificance = b
df_variants.ClinicalSignificance = df_variants.ClinicalSignificance.astype('str')

In [111]:
# df_variants.ClinicalSignificance.value_counts()
df_variants.iloc[df_variants.ClinicalSignificance.values == 'Pathogenic']

,key,chr,pos,ref,alt,gene,mutation_type,gnomad_AF,ClinicalSignificance
557085,1_152284040_CT_C,1,152284040,CT,C,FLG,frameshift,0.000669,Pathogenic
571680,17_7578265_A_G,17,7578265,A,G,TP53,missense,NaN,Pathogenic
575966,20_31023478_ACT_A,20,31023478,ACT,A,ASXL1,frameshift,NaN,Pathogenic
583015,5_176939356_CTG_C,5,176939356,CTG,C,DDX41,frameshift,NaN,Pathogenic
583070,2_71816721_A_G,2,71816721,A,G,DYSF,splice_acceptor,NaN,Pathogenic
583095,2_71817410_CTT_C,2,71817410,CTT,C,DYSF,frameshift,0.000032,Pathogenic
583097,5_13829618_C_T,5,13829618,C,T,DNAH5,splice_donor,NaN,Pathogenic
583150,2_71887766_AG_A,2,71887766,AG,A,DYSF,frameshift,NaN,Pathogenic
583170,2_71743371_CG_C,2,71743371,CG,C,DYSF,"splice,frameshift",NaN,Pathogenic
583182,1_152276135_G_A,1,152276135,G,A,FLG,stop_gained,0.000064,Pathogenic


In [114]:
df_clinvar_sub.to_csv('/path/to/local_data/gnomad_db/clinvar_subset.tsv', sep='\t',  index=False)